# F1 Pit Stop Prediction — Training Framework

Binary classification: predict `PitNextLap` (whether the driver pits on the next lap).

Pipeline: load → EDA → preprocess → CV training → submission.

EDA lives in `eda.ipynb`. This notebook covers feature engineering, cross-validated training, and submission.

In [1]:
# =============================================================
# AUTO-INSTALL + IMPORTS
# =============================================================

import sys
import subprocess

REQUIRED_PACKAGES = [
    "numpy",
    "pandas",
    "matplotlib",
    "seaborn",
    "scikit-learn",
    "lightgbm",
    "xgboost",
    "catboost",
    "scipy"
]

for package in REQUIRED_PACKAGES:
    try:
        __import__(package.replace("-", "_"))
    except ImportError:
        print(f"Installing {package}...")
        subprocess.check_call(
            [sys.executable, "-m", "pip", "install", package]
        )

# =============================================================
# IMPORTS
# =============================================================

import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import seaborn as sns

from pathlib import Path

from sklearn.model_selection import StratifiedGroupKFold

from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    log_loss
)

import lightgbm as lgb
import xgboost as xgb
import catboost as cb

from scipy.stats import rankdata

# =============================================================
# SETTINGS
# =============================================================

pd.set_option('display.max_columns', 100)

RANDOM_STATE = 42

DATA_DIR = Path('.')

print("All libraries imported successfully.")

/home/propar/.local/lib/python3.10/site-packages/matplotlib/projections/__init__.py:63: UserWarning: Unable to import Axes3D. This may be due to multiple versions of Matplotlib being installed (e.g. as a system package and as a pip package). As a result, the 3D projection is not available.
  warnings.warn("Unable to import Axes3D. This may be due to multiple versions of "


Installing scikit-learn...
Defaulting to user installation because normal site-packages is not writeable
All libraries imported successfully.


## 1. Load data

In [2]:
train = pd.read_csv(DATA_DIR / 'train.csv')
test = pd.read_csv(DATA_DIR / 'test.csv')
sample_submission = pd.read_csv(DATA_DIR / 'sample_submission.csv')

TARGET = 'PitNextLap'
ID_COL = 'id'

print('train:', train.shape, '| test:', test.shape)
train.head()

train: (439140, 16) | test: (188165, 15)


,id,Driver,Compound,Race,Year,PitStop,LapNumber,Stint,TyreLife,Position,LapTime (s),LapTime_Delta,Cumulative_Degradation,RaceProgress,Position_Change,PitNextLap
0,0,D109,HARD,Canadian Grand Prix,2022,0,50,2,39.0,8,78.491,-7.564,21.019,0.714286,5.0,1.0
1,1,D086,HARD,Dutch Grand Prix,2025,1,27,2,7.0,4,75.095,-32.617,-223.207,0.346154,-3.0,0.0
2,2,ZON,HARD,Austrian Grand Prix,2022,0,59,3,22.0,13,70.945,-7.540,-100.529,0.819444,3.0,1.0
3,3,SPE,MEDIUM,Pre-Season Testing,2023,0,2,1,2.0,7,94.361,-7.324,-7.324,0.076923,0.0,0.0
4,4,D019,HARD,Azerbaijan Grand Prix,2022,1,26,3,6.0,2,107.878,8.965,-14.139,0.361111,3.0,0.0


## 2. Data cleaning

No cleaning is applied — T35 (XGB + CatBoost) and T43 (Transformer) both train on the raw `train.csv` rows. Only minimal dtype coercions are done here.

In [3]:
# =============================================================
# DATA CLEANING — none. Only minimal dtype coercions used downstream.
# =============================================================
if 'Year' in train.columns:
    train['Year'] = train['Year'].astype('int32')
    test['Year']  = test['Year'].astype('int32')
if 'Stint' in train.columns:
    train['Stint'] = pd.to_numeric(train['Stint'], errors='coerce').astype('Int32')
    test['Stint']  = pd.to_numeric(test['Stint'],  errors='coerce').astype('Int32')

print('train shape:', train.shape, '| test shape:', test.shape)

train shape: (439140, 16) | test shape: (188165, 15)


## 3. Feature schema

T35 (XGB + CatBoost) uses the **raw README baseline only** — 10 numeric columns + 3 categoricals. No engineered features, no target encoding.

T43 (Transformer, cell 4b below) adds `LapNumber` to the numerics and a 19-feature engineered block. Those engineered features are built **inside** the transformer cell so the GBDT pipeline stays untouched, exactly mirroring the scripts in `scripts/`.

In [4]:
# =============================================================
# FEATURE SCHEMA (T35 baseline — raw features, no engineering)
# =============================================================
# Mirrors scripts/trial35.py exactly: 10 numerics + 3 categoricals.
# LapNumber is dropped per README baseline. The transformer cell below
# re-introduces LapNumber and engineered features for its own model only.

NUM_COLS = [
    'Year', 'PitStop', 'Stint', 'TyreLife', 'Position',
    'LapTime (s)', 'LapTime_Delta', 'Cumulative_Degradation',
    'RaceProgress', 'Position_Change',
]
CAT_COLS = ['Driver', 'Compound', 'Race']
FEAT_COLS = NUM_COLS + CAT_COLS

assert TARGET not in FEAT_COLS
assert 'LapNumber' not in FEAT_COLS
assert ID_COL not in FEAT_COLS

# Shared categorical codes across train+test (T35 setup).
df_all = pd.concat([train[FEAT_COLS], test[FEAT_COLS]], ignore_index=True)
for c in CAT_COLS:
    df_all[c] = df_all[c].astype(str).astype('category')
X_train = df_all.iloc[:len(train)].copy().reset_index(drop=True)
X_test  = df_all.iloc[len(train):].copy().reset_index(drop=True)

print(f'{len(NUM_COLS)} numeric + {len(CAT_COLS)} categorical = {len(FEAT_COLS)} features')
print(FEAT_COLS)

10 numeric + 3 categorical = 13 features
['Year', 'PitStop', 'Stint', 'TyreLife', 'Position', 'LapTime (s)', 'LapTime_Delta', 'Cumulative_Degradation', 'RaceProgress', 'Position_Change', 'Driver', 'Compound', 'Race']


## 4. Cross-validation: XGBoost + CatBoost (T35)

Group by `(Race, Year, Driver)` so the same stint doesn't leak across folds. `StratifiedGroupKFold(5, shuffle=True, random_state=42)` — identical to `scripts/trial35.py`.

- **XGBoost**: `max_depth=7`, `lr=0.03`, `n_estimators=6000`, `early_stopping_rounds=200`, `reg_alpha=0.1`, `reg_lambda=1.0`, `min_child_weight=20`, `subsample=0.9`, `colsample_bytree=0.9`, `tree_method='hist'`, `device='cuda'`. 2-seed bag `[42, 2024]`.
- **CatBoost**: `depth=8`, `lr=0.03`, `iterations=6000`, `l2_leaf_reg=5.0`, `bootstrap_type='Bernoulli'`, `subsample=0.9`, `task_type='GPU'`, `od_wait=200`. 2-seed bag `[42, 2024]`.

Writes XGB+CB rank-average to `submission.csv` (overwritten by the 3-way blend in cell 4b).

In [5]:
# =============================================================
# T35 — XGBoost + CatBoost (GPU), raw features only
# =============================================================
# Mirrors scripts/trial35.py: 2-seed bag for each model, 5-fold
# StratifiedGroupKFold, group = "{Race}_{Year}_{Driver}". Writes the
# OOF/test arrays (oof_xgb / oof_cb / test_xgb / test_cb) consumed by
# the transformer cell below for the 3-way rank-average ensemble.

import time
from catboost import CatBoostClassifier, Pool

y = train[TARGET].astype(np.int8).values
groups = (train['Race'].astype(str) + '_' +
          train['Year'].astype(str)  + '_' +
          train['Driver'].astype(str)).values

N_SPLITS = 5
cv = StratifiedGroupKFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE)
splits = list(cv.split(X_train, y, groups))
print(f'CV folds: {[(len(tr), len(va)) for tr, va in splits]}')

def rank_avg(*preds):
    n = len(preds[0])
    return sum(rankdata(p) / n for p in preds) / len(preds)

# ----- XGBoost: integer-coded categoricals (T35 setup) -----
X_train_xgb = X_train.copy()
X_test_xgb  = X_test.copy()
for c in CAT_COLS:
    X_train_xgb[c] = X_train_xgb[c].cat.codes.astype(np.int32)
    X_test_xgb[c]  = X_test_xgb[c].cat.codes.astype(np.int32)

XGB_SEEDS = [42, 2024]
oof_xgb  = np.zeros(len(train), dtype=np.float64)
test_xgb = np.zeros(len(test),  dtype=np.float64)
t_xgb = time.time()
print('\n=== XGBoost (T35) ===')
for fold, (tr_idx, va_idx) in enumerate(splits):
    fold_va = np.zeros(len(va_idx), dtype=np.float64)
    fold_te = np.zeros(len(test),   dtype=np.float64)
    for seed in XGB_SEEDS:
        model = xgb.XGBClassifier(
            max_depth=7, learning_rate=0.03, n_estimators=6000,
            early_stopping_rounds=200, reg_alpha=0.1, reg_lambda=1.0,
            min_child_weight=20, subsample=0.9, colsample_bytree=0.9,
            tree_method='hist', device='cuda',
            eval_metric='auc', random_state=seed, verbosity=0,
        )
        model.fit(X_train_xgb.iloc[tr_idx], y[tr_idx],
                  eval_set=[(X_train_xgb.iloc[va_idx], y[va_idx])], verbose=False)
        fold_va += model.predict_proba(X_train_xgb.iloc[va_idx])[:, 1]
        fold_te += model.predict_proba(X_test_xgb)[:, 1]
    fold_va /= len(XGB_SEEDS); fold_te /= len(XGB_SEEDS)
    oof_xgb[va_idx] = fold_va
    test_xgb += fold_te / N_SPLITS
    print(f'  fold {fold}  va_auc={roc_auc_score(y[va_idx], fold_va):.5f}  '
          f't={time.time()-t_xgb:.0f}s')
xgb_auc = roc_auc_score(y, oof_xgb)
print(f'XGB OOF AUC = {xgb_auc:.5f}  ({time.time()-t_xgb:.0f}s)')

# ----- CatBoost: string categoricals on GPU (T35 setup) -----
X_train_cb = X_train.copy()
X_test_cb  = X_test.copy()
for c in CAT_COLS:
    X_train_cb[c] = X_train_cb[c].astype(str)
    X_test_cb[c]  = X_test_cb[c].astype(str)
cb_cat_idx = [X_train_cb.columns.get_loc(c) for c in CAT_COLS]

CB_SEEDS = [42, 2024]
oof_cb  = np.zeros(len(train), dtype=np.float64)
test_cb = np.zeros(len(test),  dtype=np.float64)
t_cb = time.time()
print('\n=== CatBoost (T35) ===')
for fold, (tr_idx, va_idx) in enumerate(splits):
    fold_va = np.zeros(len(va_idx), dtype=np.float64)
    fold_te = np.zeros(len(test),   dtype=np.float64)
    tr_pool = Pool(X_train_cb.iloc[tr_idx], label=y[tr_idx], cat_features=cb_cat_idx)
    va_pool = Pool(X_train_cb.iloc[va_idx], label=y[va_idx], cat_features=cb_cat_idx)
    te_pool = Pool(X_test_cb,                                 cat_features=cb_cat_idx)
    for seed in CB_SEEDS:
        model = CatBoostClassifier(
            depth=8, learning_rate=0.03, iterations=6000,
            l2_leaf_reg=5.0, eval_metric='AUC',
            bootstrap_type='Bernoulli', subsample=0.9,
            task_type='GPU', devices='0',
            random_seed=seed, od_type='Iter', od_wait=200,
            verbose=False, allow_writing_files=False,
        )
        model.fit(tr_pool, eval_set=va_pool, use_best_model=True)
        fold_va += model.predict_proba(va_pool)[:, 1]
        fold_te += model.predict_proba(te_pool)[:, 1]
    fold_va /= len(CB_SEEDS); fold_te /= len(CB_SEEDS)
    oof_cb[va_idx] = fold_va
    test_cb += fold_te / N_SPLITS
    print(f'  fold {fold}  va_auc={roc_auc_score(y[va_idx], fold_va):.5f}  '
          f't={time.time()-t_cb:.0f}s')
cb_auc = roc_auc_score(y, oof_cb)
print(f'CB OOF AUC = {cb_auc:.5f}  ({time.time()-t_cb:.0f}s)')

# ----- XGB + CB rank-average (preliminary submission, T35 partial) -----
oof_ensemble  = rank_avg(oof_xgb,  oof_cb)
test_ensemble = rank_avg(test_xgb, test_cb)

print('\n================ XGB + CB ENSEMBLE ================')
print(f'XGB    OOF AUC : {xgb_auc:.5f}')
print(f'CB     OOF AUC : {cb_auc:.5f}')
print(f'XGB+CB OOF AUC : {roc_auc_score(y, oof_ensemble):.5f}')
print(f'XGB+CB OOF AP  : {average_precision_score(y, oof_ensemble):.5f}')
print(f'XGB+CB OOF LL  : {log_loss(y, oof_ensemble):.5f}')

submission = pd.DataFrame({ID_COL: test[ID_COL], TARGET: test_ensemble})
submission.to_csv('submission.csv', index=False)
print('\nsubmission.csv saved (XGB + CB rank-average — will be overwritten by 3-way blend)')

CV folds: [(350258, 88882), (350518, 88622), (351246, 87894), (352601, 86539), (351937, 87203)]

=== XGBoost (T35) ===
  fold 0  va_auc=0.94958  t=13s
  fold 1  va_auc=0.94855  t=24s
  fold 2  va_auc=0.94781  t=36s
  fold 3  va_auc=0.94898  t=47s
  fold 4  va_auc=0.94942  t=58s
XGB OOF AUC = 0.94886  (58s)

=== CatBoost (T35) ===


Default metric period is 5 because AUC is/are not implemented for GPU
Default metric period is 5 because AUC is/are not implemented for GPU


  fold 0  va_auc=0.95005  t=316s


Default metric period is 5 because AUC is/are not implemented for GPU
Default metric period is 5 because AUC is/are not implemented for GPU


  fold 1  va_auc=0.94882  t=676s


...

  fold 2  va_auc=0.94775  t=1041s


Default metric period is 5 because AUC is/are not implemented for GPU
Default metric period is 5 because AUC is/are not implemented for GPU


  fold 3  va_auc=0.94886  t=1323s


Default metric period is 5 because AUC is/are not implemented for GPU
Default metric period is 5 because AUC is/are not implemented for GPU


  fold 4  va_auc=0.94962  t=1680s
CB OOF AUC = 0.94901  (1680s)

================ XGB + CB ENSEMBLE ================
XGB    OOF AUC : 0.94886
CB     OOF AUC : 0.94901
XGB+CB OOF AUC : 0.94972
XGB+CB OOF AP  : 0.81628
XGB+CB OOF LL  : 0.55474

submission.csv saved (XGB + CB rank-average — will be overwritten by 3-way blend)


## 4b. Transformer (T43) + 3-way ensemble

Per-stint Transformer encoder with concat-fusion categoricals (`Driver=24`, `Compound=6`, `Race=12`), causal self-attention over `(Year, Race, Driver)` sequences, and an auxiliary head that predicts next-lap `TyreLife` (MSE, weight=0.3). 6-seed bag × 5-fold StratifiedGroupKFold; OOF is read off the train rows of each fold's validation sequences. The final submission is the 3-way rank-average of XGB + CatBoost + Transformer — T43, CV OOF AUC 0.95018, current best in `trials.csv`.

Requires a CUDA GPU and PyTorch. Reuses `oof_xgb`/`oof_cb`/`test_xgb`/`test_cb` from the cell above.

In [6]:
# =============================================================
# TRANSFORMER (T43) + 3-WAY ENSEMBLE
# =============================================================
# Sequence model (per Year/Race/Driver) with causal self-attention,
# concat-fusion categoricals, and an auxiliary head predicting
# next-lap TyreLife. 6-seed bag x 5-fold; OOF is then rank-averaged
# with the XGB and CatBoost OOFs from the previous cell.

import time, warnings
warnings.filterwarnings('ignore')

import torch
import torch.nn as nn
import torch.nn.functional as F

assert torch.cuda.is_available(), 'T43 transformer requires a CUDA GPU'
DEVICE = torch.device('cuda')
torch.set_num_threads(8)
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True
torch.backends.cudnn.benchmark = True
print(f'device = {torch.cuda.get_device_name(0)}')

AUX_WEIGHT = 0.3
TFM_SEEDS  = [42, 2024, 7, 31337, 1, 99]
EPOCHS, BATCH, LR, PATIENCE = 35, 256, 2e-3, 8

# --- Build a combined train+test frame with per-stint sequences --------------
RAW_NUM = ['Year','LapNumber','PitStop','Stint','TyreLife','Position',
           'LapTime (s)','LapTime_Delta','Cumulative_Degradation',
           'RaceProgress','Position_Change']
TFM_CAT = ['Driver','Compound','Race']

train_tag = train.copy(); train_tag['_is_train'] = 1
test_tag  = test.copy();  test_tag['_is_train']  = 0
test_tag[TARGET] = -1
df = pd.concat([train_tag, test_tag], ignore_index=True)
df = df.sort_values(['Year','Race','Driver','LapNumber']).reset_index(drop=True)

eps = 1e-6
df['RemainingRace']           = 1.0 - df['RaceProgress']
df['PitWindow']               = df['RaceProgress'] * (1.0 - df['RaceProgress'])
df['IsLateRace']              = (df['RaceProgress'] > 0.75).astype(np.float32)
df['LapTime_per_TyreLife']    = df['LapTime (s)'] / (df['TyreLife'] + eps)
df['Deg_per_TyreLife']        = df['Cumulative_Degradation'] / (df['TyreLife'] + eps)
df['TyreStress']              = df['TyreLife'] * df['Cumulative_Degradation']
df['StrategicUrgency']        = df['TyreStress'] * df['RemainingRace']
df['TyreExhaustion']          = (df['TyreLife'] ** 2) * df['RemainingRace']
df['TyreCliff']               = df['TyreLife'] * df['LapTime_Delta']
df['PositionPressure']        = df['Position'] * df['RemainingRace']
df['RecoveryPressure']        = df['Position_Change'].abs() * df['Cumulative_Degradation']
df['_CompoundCode']           = pd.Categorical(df['Compound'].astype(str)).codes.astype(np.float32)
df['CompoundTyreInteraction'] = df['_CompoundCode'] * df['TyreLife']
df['StrategyPressure']        = df['TyreStress'] * df['PitWindow']
df['PitOffsetPotential']      = df['LapTime_Delta'] * df['RemainingRace'] * 10.0
df['UndercutPotential']       = df['LapTime_Delta'] * df['Position_Change'].abs() * df['RemainingRace']
df['StintSurvivalPressure']   = df['TyreLife'] * df['RemainingRace'] * df['LapTime_Delta']
df['PaceCollapse']            = df['LapTime_Delta'] * df['Cumulative_Degradation']
df['LateRaceTyreRisk']        = df['IsLateRace'] * df['TyreLife']

ENG_NUM = ['RemainingRace','PitWindow','IsLateRace','LapTime_per_TyreLife',
           'Deg_per_TyreLife','TyreStress','StrategicUrgency','TyreExhaustion',
           'TyreCliff','PositionPressure','RecoveryPressure','_CompoundCode',
           'CompoundTyreInteraction','StrategyPressure','PitOffsetPotential',
           'UndercutPotential','StintSurvivalPressure','PaceCollapse','LateRaceTyreRisk']
NUM_COLS = RAW_NUM + ENG_NUM
TYRELIFE_IDX = NUM_COLS.index('TyreLife')
print(f'transformer numerics: {len(NUM_COLS)}  ({len(RAW_NUM)} raw + {len(ENG_NUM)} engineered)')

for c in NUM_COLS:
    s = df[c]
    if s.dtype == 'O':
        s = pd.to_numeric(s, errors='coerce')
    s = s.replace([np.inf, -np.inf], np.nan).fillna(0.0)
    df[c] = s.astype(np.float32)
for c in NUM_COLS:
    mu = df[c].mean(); sd = df[c].std() + 1e-6
    df[c] = ((df[c] - mu) / sd).astype(np.float32)

cat_vocab = {}
for c in TFM_CAT:
    cats = pd.Categorical(df[c].astype(str))
    df[c + '_id'] = cats.codes.astype(np.int64)
    cat_vocab[c] = len(cats.categories)

df['_seq'] = df.groupby(['Year','Race','Driver']).ngroup()
N_SEQ = int(df['_seq'].max() + 1)
MAX_LEN = int(df.groupby('_seq').size().max())

num_arr    = np.zeros((N_SEQ, MAX_LEN, len(NUM_COLS)), dtype=np.float32)
cat_arr    = np.zeros((N_SEQ, MAX_LEN, len(TFM_CAT)), dtype=np.int64)
pos_arr    = np.zeros((N_SEQ, MAX_LEN), dtype=np.int64)
y_arr      = np.zeros((N_SEQ, MAX_LEN), dtype=np.float32)
mask_arr   = np.zeros((N_SEQ, MAX_LEN), dtype=np.float32)
loss_mask  = np.zeros((N_SEQ, MAX_LEN), dtype=np.float32)
aux_target = np.zeros((N_SEQ, MAX_LEN), dtype=np.float32)
aux_mask   = np.zeros((N_SEQ, MAX_LEN), dtype=np.float32)

cat_id_cols = [c + '_id' for c in TFM_CAT]
seq_train_pos = [None] * N_SEQ; seq_train_idx = [None] * N_SEQ
seq_test_pos  = [None] * N_SEQ; seq_test_idx  = [None] * N_SEQ
train_id_to_pos = {tid: i for i, tid in enumerate(train[ID_COL].values)}
test_id_to_pos  = {tid: i for i, tid in enumerate(test[ID_COL].values)}

for sid, g in df.groupby('_seq'):
    L = len(g)
    num_arr[sid, :L]   = g[NUM_COLS].values
    cat_arr[sid, :L]   = g[cat_id_cols].values
    pos_arr[sid, :L]   = g['LapNumber'].values.clip(0, 199)
    y_arr[sid, :L]     = np.where(g['_is_train'].values == 1, g[TARGET].values, 0.0)
    mask_arr[sid, :L]  = 1.0
    loss_mask[sid, :L] = g['_is_train'].values.astype(np.float32)
    if L > 1:
        aux_target[sid, :L-1] = num_arr[sid, 1:L, TYRELIFE_IDX]
        aux_mask[sid, :L-1]   = 1.0
    gr = g.reset_index(drop=True)
    is_tr = gr['_is_train'].values == 1
    seq_train_pos[sid] = np.where(is_tr)[0].astype(np.int64)
    seq_test_pos[sid]  = np.where(~is_tr)[0].astype(np.int64)
    seq_train_idx[sid] = np.array([train_id_to_pos[t] for t in gr.loc[is_tr, ID_COL].values], dtype=np.int64)
    seq_test_idx[sid]  = np.array([test_id_to_pos[t]  for t in gr.loc[~is_tr, ID_COL].values], dtype=np.int64)

num_g   = torch.from_numpy(num_arr).to(DEVICE)
cat_g   = torch.from_numpy(cat_arr).to(DEVICE)
pos_g   = torch.from_numpy(pos_arr).to(DEVICE)
y_g     = torch.from_numpy(y_arr).to(DEVICE)
m_g     = torch.from_numpy(mask_arr).to(DEVICE)
lm_g    = torch.from_numpy(loss_mask).to(DEVICE)
aux_t_g = torch.from_numpy(aux_target).to(DEVICE)
aux_m_g = torch.from_numpy(aux_mask).to(DEVICE)


class PitTransformerMT(nn.Module):
    def __init__(self, num_cols, cat_vocab,
                 num_dim_per_col=6,
                 cat_dims=None,
                 d_model=128, n_heads=4, n_layers=4, dim_ff=256,
                 dropout=0.1, max_pos=200):
        super().__init__()
        if cat_dims is None:
            cat_dims = {'Driver': 24, 'Compound': 6, 'Race': 12}
        self.cat_keys = list(cat_vocab.keys())
        self.num_proj = nn.Linear(len(num_cols), len(num_cols) * num_dim_per_col)
        num_total = len(num_cols) * num_dim_per_col
        self.cat_emb = nn.ModuleDict({k: nn.Embedding(cat_vocab[k], cat_dims[k]) for k in self.cat_keys})
        cat_total = sum(cat_dims[k] for k in self.cat_keys)
        self.fuse = nn.Linear(num_total + cat_total, d_model)
        self.pos_emb = nn.Embedding(max_pos, d_model)
        self.drop = nn.Dropout(dropout)
        enc_layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=n_heads, dim_feedforward=dim_ff,
            dropout=dropout, batch_first=True, activation='gelu', norm_first=True)
        self.encoder = nn.TransformerEncoder(enc_layer, num_layers=n_layers)
        self.head_norm = nn.LayerNorm(d_model)
        self.head_pit  = nn.Linear(d_model, 1)
        self.head_aux  = nn.Linear(d_model, 1)

    def forward(self, num, cats, pos, key_padding_mask, attn_mask):
        n = self.num_proj(num)
        emb_list = [self.cat_emb[k](cats[..., i]) for i, k in enumerate(self.cat_keys)]
        fused = torch.cat([n] + emb_list, dim=-1)
        h = self.fuse(fused) + self.pos_emb(pos)
        h = self.drop(h)
        h = self.encoder(h, mask=attn_mask, src_key_padding_mask=key_padding_mask)
        h = self.head_norm(h)
        return self.head_pit(h).squeeze(-1), self.head_aux(h).squeeze(-1)


def _causal_mask(L, device):
    return torch.triu(torch.full((L, L), float('-inf'), device=device), diagonal=1)


# Sequence-level CV that matches the row-level CV used for XGB/CB
seq_first = df.groupby('_seq').head(1).reset_index(drop=True)
seq_groups = (seq_first['Race'].astype(str) + '_' + seq_first['Year'].astype(str) + '_' + seq_first['Driver'].astype(str)).values
seq_y = (y_arr.sum(axis=1) > 0).astype(int)
cv_seq = StratifiedGroupKFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE)
seq_splits = list(cv_seq.split(np.arange(N_SEQ), seq_y, seq_groups))

causal = _causal_mask(MAX_LEN, DEVICE)
y_full = train[TARGET].astype(np.int8).values
tfm_oof  = np.zeros(len(train), dtype=np.float64)
tfm_test = np.zeros(len(test),  dtype=np.float64)

t0 = time.time()
for fold, (tr_seq, va_seq) in enumerate(seq_splits):
    fstart = time.time()
    print(f'\n--- TFM fold {fold} ---')
    fold_oof  = np.zeros(len(train), dtype=np.float64)
    fold_test = np.zeros(len(test),  dtype=np.float64)
    for seed in TFM_SEEDS:
        torch.manual_seed(seed + fold); np.random.seed(seed + fold)
        model = PitTransformerMT(NUM_COLS, cat_vocab).to(DEVICE)
        opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)
        sch = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=EPOCHS)
        scaler = torch.amp.GradScaler('cuda')
        best_va, best_state, best_ep, pat = 0.0, None, -1, 0
        for ep in range(EPOCHS):
            model.train(); perm = np.random.permutation(tr_seq)
            for i in range(0, len(perm), BATCH):
                b = torch.from_numpy(perm[i:i+BATCH]).to(DEVICE)
                num_b = num_g.index_select(0, b); cat_b = cat_g.index_select(0, b)
                pos_b = pos_g.index_select(0, b); y_b   = y_g.index_select(0, b)
                m_b   = m_g.index_select(0, b);   lm_b  = lm_g.index_select(0, b)
                at_b  = aux_t_g.index_select(0, b); am_b = aux_m_g.index_select(0, b)
                kpm   = (m_b == 0)
                opt.zero_grad(set_to_none=True)
                with torch.amp.autocast('cuda', dtype=torch.float16):
                    logits_pit, aux_pred = model(num_b, cat_b, pos_b, kpm, causal)
                    bce = F.binary_cross_entropy_with_logits(logits_pit, y_b, reduction='none')
                    loss_pit = (bce * lm_b).sum() / lm_b.sum().clamp_min(1.0)
                    aux_err  = (aux_pred - at_b) ** 2
                    loss_aux = (aux_err * am_b).sum() / am_b.sum().clamp_min(1.0)
                    loss = loss_pit + AUX_WEIGHT * loss_aux
                scaler.scale(loss).backward(); scaler.unscale_(opt)
                torch.nn.utils.clip_grad_norm_(model.parameters(), 2.0)
                scaler.step(opt); scaler.update()
            sch.step(); model.eval()
            with torch.no_grad(), torch.amp.autocast('cuda', dtype=torch.float16):
                preds_all, ys_all = [], []
                for i in range(0, len(va_seq), BATCH):
                    b = torch.from_numpy(va_seq[i:i+BATCH]).to(DEVICE)
                    num_b = num_g.index_select(0, b); cat_b = cat_g.index_select(0, b)
                    pos_b = pos_g.index_select(0, b); m_b   = m_g.index_select(0, b)
                    kpm = (m_b == 0)
                    logits_pit, _ = model(num_b, cat_b, pos_b, kpm, causal)
                    p = torch.sigmoid(logits_pit.float()).cpu().numpy()
                    lm_b = lm_g.index_select(0, b).cpu().numpy()
                    y_b  = y_g.index_select(0, b).cpu().numpy(); sel = lm_b == 1
                    preds_all.append(p[sel]); ys_all.append(y_b[sel])
                va_auc = roc_auc_score(np.concatenate(ys_all), np.concatenate(preds_all))
            if va_auc > best_va:
                best_va, best_ep, pat = va_auc, ep, 0
                best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            else:
                pat += 1
                if pat >= PATIENCE:
                    print(f'    seed={seed} stop ep{ep} best ep{best_ep} auc={best_va:.5f}')
                    break
        else:
            print(f'    seed={seed} done ep{EPOCHS-1} best ep{best_ep} auc={best_va:.5f}')
        model.load_state_dict(best_state); model.eval()
        with torch.no_grad(), torch.amp.autocast('cuda', dtype=torch.float16):
            for i in range(0, len(va_seq), BATCH):
                b_np = va_seq[i:i+BATCH]; b = torch.from_numpy(b_np).to(DEVICE)
                num_b = num_g.index_select(0, b); cat_b = cat_g.index_select(0, b)
                pos_b = pos_g.index_select(0, b); m_b   = m_g.index_select(0, b)
                logits_pit, _ = model(num_b, cat_b, pos_b, (m_b == 0), causal)
                p = torch.sigmoid(logits_pit.float()).cpu().numpy()
                for k, sid in enumerate(b_np):
                    tp = seq_train_pos[sid]; ti = seq_train_idx[sid]
                    if len(tp): fold_oof[ti] += p[k, tp]
            for i in range(0, N_SEQ, BATCH):
                b_np = np.arange(i, min(i+BATCH, N_SEQ)); b = torch.from_numpy(b_np).to(DEVICE)
                num_b = num_g.index_select(0, b); cat_b = cat_g.index_select(0, b)
                pos_b = pos_g.index_select(0, b); m_b   = m_g.index_select(0, b)
                logits_pit, _ = model(num_b, cat_b, pos_b, (m_b == 0), causal)
                p = torch.sigmoid(logits_pit.float()).cpu().numpy()
                for k, sid in enumerate(b_np):
                    tp = seq_test_pos[sid]; ti = seq_test_idx[sid]
                    if len(tp): fold_test[ti] += p[k, tp]
        del model, best_state; torch.cuda.empty_cache()
    fold_oof  /= len(TFM_SEEDS)
    fold_test /= len(TFM_SEEDS)
    tfm_oof  += fold_oof
    tfm_test += fold_test / N_SPLITS
    print(f'  fold {fold} time {time.time()-fstart:.0f}s')

tfm_auc = roc_auc_score(y_full, tfm_oof)
print(f'\nTFM standalone OOF AUC = {tfm_auc:.5f}')
print(f'TOTAL transformer time : {time.time()-t0:.0f}s')

# --- 3-way rank-average ensemble ---------------------------------------------
oof_3way  = rank_avg(oof_xgb,  oof_cb,  tfm_oof)
test_3way = rank_avg(test_xgb, test_cb, tfm_test)

print('\n================ T43 3-WAY ENSEMBLE ================')
print(f'XGB+CB OOF AUC  : {roc_auc_score(y_full, oof_ensemble):.5f}')
print(f'TFM    OOF AUC  : {tfm_auc:.5f}')
print(f'3-way  OOF AUC  : {roc_auc_score(y_full, oof_3way):.5f}')
print(f'3-way  OOF AP   : {average_precision_score(y_full, oof_3way):.5f}')
print(f'3-way  OOF LL   : {log_loss(y_full, oof_3way):.5f}')

# Overwrite submission.csv with the 3-way rank-average — T43 best model.
submission = pd.DataFrame({ID_COL: test[ID_COL], TARGET: test_3way})
submission.to_csv('submission.csv', index=False)
print('\nsubmission.csv saved (3-way rank-avg of XGB + CatBoost + Transformer)')

device = NVIDIA GeForce RTX 5060 Laptop GPU
transformer numerics: 30  (11 raw + 19 engineered)

--- TFM fold 0 ---
    seed=42 stop ep30 best ep22 auc=0.94307
    seed=2024 stop ep30 best ep22 auc=0.94278
    seed=7 stop ep29 best ep21 auc=0.94286
    seed=31337 done ep34 best ep27 auc=0.94371
    seed=1 stop ep29 best ep21 auc=0.94253
    seed=99 stop ep27 best ep19 auc=0.94271
  fold 0 time 371s

--- TFM fold 1 ---
    seed=42 stop ep33 best ep25 auc=0.94416
    seed=2024 stop ep31 best ep23 auc=0.94418
    seed=7 stop ep32 best ep24 auc=0.94401
    seed=31337 stop ep30 best ep22 auc=0.94390
    seed=1 stop ep32 best ep24 auc=0.94388
    seed=99 stop ep34 best ep26 auc=0.94444
  fold 1 time 392s

--- TFM fold 2 ---
    seed=42 stop ep29 best ep21 auc=0.94323
    seed=2024 stop ep32 best ep24 auc=0.94378
    seed=7 stop ep32 best ep24 auc=0.94341
    seed=31337 stop ep32 best ep24 auc=0.94298
    seed=1 stop ep28 best ep20 auc=0.94346
    seed=99 stop ep26 best ep18 auc=0.94344
  fold

## 5. Feature importance

In [7]:
# Feature importance is not computed in the T35/T43 setup
# (matches scripts/trial35.py and scripts/trial43.py — neither dumps importances).
# This cell is intentionally a no-op; left as a placeholder.
pass

## 6. Submission

In [8]:
# Final submission = T43 3-way rank-average (XGB + CatBoost + Transformer).
# Falls back to XGB+CB rank-average if the transformer cell hasn't been run.
try:
    final_test = test_3way
    blend_name = '3-way (XGB + CatBoost + Transformer)'
except NameError:
    final_test = test_ensemble
    blend_name = '2-way (XGB + CatBoost)'

submission = sample_submission.copy()
submission[TARGET] = final_test
submission.to_csv('submission.csv', index=False)
print(f'submission.csv written — {blend_name}')
submission.head()

submission.csv written — 3-way (XGB + CatBoost + Transformer)


,id,PitNextLap
0,439140,0.297450
1,439141,0.355796
2,439142,0.310070
3,439143,0.689177
4,439144,0.966613
